[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1O6aonBOqukQN05ZCdOHvXITX6ojQd01L)

## mBERT un XLM-RoBERTa apmācība

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    BertForSequenceClassification,
    BertTokenizer,
    Trainer,
    TrainingArguments,
    XLMRobertaForSequenceClassification,
    XLMRobertaTokenizer
)

In [ ]:
!wget -q -O data.csv https://raw.githubusercontent.com/IvoDz/lv-text-complexity/refs/heads/main/data/data.csv

In [ ]:
df = pd.read_csv("data.csv")
label_map = {"viegls": 0, "vidējs": 1, "sarežģīts": 2}
df["level"] = df["level"].map(label_map)

In [ ]:
label_counts = df["level"].value_counts().sort_index()
colors = ['green', 'yellow', 'red']
plt.figure(figsize=(6, 4))
plt.bar(label_counts.index.astype(str), label_counts.values, color=colors)
plt.xlabel("Label (0=viegls, 1=vidējs, 2=sarežģīts)")
plt.ylabel("Skaits")
plt.title("Klašu sadalījums")
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["level"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["level"], random_state=42)

### mBERT
# model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=3)
# tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

model = XLMRobertaForSequenceClassification.from_pretrained('xlm-roberta-base', num_labels=3)
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

ds = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(val_df),
    'test': Dataset.from_pandas(test_df),
})

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

ds = ds.map(tokenize, batched=True)
ds = ds.remove_columns(['text'])
ds = ds.rename_column('level', 'labels')
ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)

training_args = TrainingArguments(
    report_to=[],
    output_dir='./classifier',
    eval_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_dir='./logs',
    logging_steps=10,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.tensor(logits).argmax(dim=1).cpu().numpy()
    labels = torch.tensor(labels).cpu().numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': f1,
    }


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
eval_results = trainer.evaluate(ds["test"])
print("Test Accuracy:", eval_results["eval_accuracy"])

Saglabā modeli

In [ ]:
trainer.save_model("roberta-0719")
tokenizer.save_pretrained("roberta-0719")

Negatīvie piemēri

In [ ]:
predictions = trainer.predict(ds["test"])
logits = predictions.predictions
labels = predictions.label_ids
preds = np.argmax(logits, axis=1)

wrong_indices = np.where(preds != labels)[0]
print(f"Total misclassified examples: {len(wrong_indices)}")

for idx in wrong_indices:
    print("Text:", test_df.iloc[idx]['text'])
    print("True Label:", labels[idx])
    print("Predicted Label:", preds[idx])
    print("---")